Simple RNN

## LSTM for multi variable of stock


In [ ]:
import numpy as np 
import pandas as pd
## data scrape package
import pandas_datareader.data as web
import matplotlib.pyplot as plt
#scaling and preprocessing
from sklearn.preprocessing import MinMaxScaler
# keras network @ https://www.tensorflow.org/guide/keras/rnn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import yfinance as yf
# set Random seed for reproducibility
np.random.seed(2505)


# Define the ticker symbol for  futures prediction
ticker = 'IONQ'

# Download 5 years of daily historical data
ticker = yf.download(ticker, period="730d", interval="1h",prepost=True)
ticker.columns = ticker.columns.droplevel('Ticker')
# Strip the timezone completely (makes it timezone-naive)
ticker.index = ticker.index.tz_localize(None)

# Now convert to Unix epoch seconds
ticker.index = ticker.index.astype('int64') // 10**9
new_order = [col for col in ticker.columns if col != 'Close'] + ['Close']

# Reorder the DataFrame columns
ticker = ticker[new_order]

# Display the first few rows
print(ticker.tail())


print(ticker.head ())
print(ticker.columns)# date is an index for time series

plt.plot(ticker['Close'])


ticker.describe()
#Fractional change from the previous in time series rows
fractional_change = ticker['Close'].pct_change()
column_names=ticker.columns

x=ticker.values
minmax_scale=MinMaxScaler()
x_scaled=minmax_scale.fit_transform(x)
df=pd.DataFrame(x_scaled)

df.columns=column_names
print(df.columns)

df_ticker_ptc=df
df_ticker_ptc.head()

plt.plot(fractional_change)
fractional_change.hist()


# Need the data to be in the form [sample, time steps, features (dimension of each element)]
samples = 10 # Number of samples (in past)
steps = 1 # Number of steps (in future)
X = [] # X array
Y = [] # Y array
for i in range(df_ticker_ptc.shape[0] - samples):
    X.append(df_ticker_ptc.iloc[i:i+samples, 0:4].values) # Independent Samples with multiple variable 
    Y.append(df_ticker_ptc.iloc[i+samples, 4:].values) # Dependent Samples last column which is (close for predication)
print('Training Data: Length is ', len(X[0:1][0]), ': ', X[0:1])
print('Testing Data: Length is ', len(Y[0:1]), ': ', Y[0:1])

# Reshape the data so that the inputs will be acceptable to the model
X=np.array(X)
Y=np.array(Y)
print('Dimension of X',X.shape, 'Dimension of Y',Y.shape)
threshold =round(0.9*X.shape[0])
print(threshold)

# Get the training and testing set
threshold=round(0.9*X.shape[0])
trainX = X[:threshold]
trainY = Y[:threshold]
testX = X[threshold:]
testY = Y[threshold:]
print('Traing Lenght',trainX.shape, trainY.shape, 'Testing Lenght',testX.shape, testY.shape)



# Build again the LSTM Model
model = keras.Sequential()

# Swap SimpleRNN for LSTM while preserving units, activation, and your input shape
model.add(layers.LSTM(units=30, activation='tanh', use_bias=True, input_shape=(trainX.shape[1], trainX.shape[2])))

# Add a dropout layer (penalizing more complex models) -- prevents overfitting 
model.add(layers.Dropout(rate=0.2))

# Add a Dense layer with 1 units (since we are doing a regression task)
model.add(layers.Dense(1))

# Evaluating loss function of MSE using the adam optimizer
model.compile(loss='mse', optimizer='adam', metrics=['mae', 'mape'])

# Print out architecture
model.summary()

history=model.fit(X[:threshold],
                  Y[:threshold],
                  shuffle=False, # Because this is the time series data set
                  epochs=100,
                  batch_size=32,
                  validation_split=0.20,
                  verbose=1)

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('LSTM RNN loss with multi variables')
plt.ylabel('loss')
plt.xlabel('Number of Epoch')
plt.legend(['train', 'test'], loc='upper right')
# plt.ylim(0, 0.3) # <-- Cuts off the giant early losses to focus on the tail
# Note:
# if training loss >> validation loss -> Underfitting
# if training loss << validation loss -> Overfitting (i.e model is smart enough to have mapped the entire dataset..)
# Several ways to address overfitting:
# Reduce complexity of model (hidden layers, neurons, parameters input etc)
# Add dropout and tune rate
# More data :)

y_pred=model.predict(testX)
plt.plot(testY,label='True Value')
plt.plot(y_pred,label='Forecasted Value')
plt.legend()
plt.ylim(.33, .86)